## Leetcode 178. Rank Scores

In [0]:
data = [(1,3.50),(2,3.65),(3,4.00),(4,3.85),(5,4.00),(6,3.65)]
Scores = spark.createDataFrame(data, ['id','score'])
Scores.createOrReplaceTempView('Scores')


Scores
| id | score |
|----|-------|
| 1  | 3.50  |
| 2  | 3.65  |
| 3  | 4.00  |
| 4  | 3.85  |
| 5  | 4.00  |
| 6  | 3.65  |


### Spark SQL

In [0]:
spark.sql('''
          select score,
dense_rank() over (order by score desc) as rank
from Scores''').show()

### Spark DataFrame API

In [0]:
from pyspark.sql.functions import col, dense_rank
from pyspark.sql.window import Window
from pyspark.sql.types import *

In [0]:
window= Window.orderBy(col('score').desc())
Scores= Scores.withColumn('rank',dense_rank().over(window))

Scores.select(col('score'), col('rank')).show()

### In Production

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", "true") # AQE to decide partitions on its own

window = Window.orderBy(col('score').desc()) \
               .rowsBetween(Window.unboundedPreceding, Window.currentRow) # only to suppress warning

Scores.withColumn('rank', dense_rank().over(window)) \
      .select(col('score'), col('rank')) \
      .orderBy(col('score').desc()) \
      .show()